**CI twin of `ch02-activation-functions.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
W1, b1 = rng.normal(size=(3, 2)), rng.normal(size=3)
W2, b2 = rng.normal(size=(1, 3)), rng.normal(size=1)

def two_linear_layers(x):
    hidden = W1 @ x + b1          # layer 1: three weighted sums
    return W2 @ hidden + b2       # layer 2: a weighted sum of those

# The algebra: W2·(W1·x + b1) + b2  =  (W2·W1)·x + (W2·b1 + b2)
W_merged = W2 @ W1
b_merged = W2 @ b1 + b2

worst = 0.0
for probe in rng.normal(size=(5, 2)):
    gap = float(np.abs(two_linear_layers(probe)
                       - (W_merged @ probe + b_merged)).max())
    worst = max(worst, gap)
print(f"largest disagreement across 5 random inputs: {worst}")
print(f"merged layer:  W = {np.round(W_merged, 3)},  b = {np.round(b_merged, 3)}")

In [ ]:
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 300)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)

slopes = [sigmoid * (1 - sigmoid), 1 - tanh**2, (z > 0).astype(float)]
names = ["sigmoid", "tanh", "ReLU"]

fig, axes = plt.subplots(2, 3, figsize=(8, 4), sharex=True)
for j, (f, s, name) in enumerate(zip([sigmoid, tanh, relu], slopes, names)):
    axes[0, j].plot(z, f); axes[0, j].set_title(name, fontsize=9)
    axes[1, j].plot(z, s, color="crimson")
    axes[1, j].set_title(f"{name}'s slope", fontsize=8)
plt.tight_layout()
plt.show()

sig = lambda v: 1 / (1 + np.exp(-v))
print(f"sigmoid slope at z=0:  {float(sig(0) * (1 - sig(0))):.2f} (its max)")
print(f"sigmoid slope at z=10: {float(sig(10) * (1 - sig(10))):.7f}")

In [ ]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

Xc, yc = make_circles(n_samples=400, factor=0.4, noise=0.08, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(
    Xc, yc, test_size=0.25, random_state=42, stratify=yc)

one_neuron = LogisticRegression().fit(Xtr, ytr)
print(f"single neuron: {accuracy_score(yte, one_neuron.predict(Xte)):.3f}")

In [ ]:
from sklearn.neural_network import MLPClassifier

net = MLPClassifier(hidden_layer_sizes=(8,), activation="relu",
                    random_state=0, max_iter=3000).fit(Xtr, ytr)
print(f"2-layer ReLU network: {accuracy_score(yte, net.predict(Xte)):.3f}")

gx, gy = np.meshgrid(np.linspace(-1.4, 1.4, 200),
                     np.linspace(-1.4, 1.4, 200))
grid = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.4))
for ax, model, title in [(axes[0], one_neuron, "single neuron: 0.500"),
                         (axes[1], net, "8 ReLU neurons: 1.000")]:
    zz = model.predict(grid).reshape(gx.shape)
    ax.contourf(gx, gy, zz, alpha=0.15, cmap="coolwarm")
    ax.scatter(Xc[:, 0], Xc[:, 1], c=yc, cmap="coolwarm", s=8)
    ax.set_title(title, fontsize=9)
plt.show()

In [ ]:
import numpy as np

W1 = np.array([[1.0, 2.0], [0.0, 1.0]])
b1 = np.array([1.0, 0.0])
W2 = np.array([[2.0, -1.0]])
b2 = np.array([3.0])

W_merged = W2 @ W1
b_merged = W2 @ b1 + b2

x = np.array([1.0, 1.0])
layered = float((W2 @ (W1 @ x + b1) + b2)[0])
merged = float((W_merged @ x + b_merged)[0])

run_tests([
    ("merged weights", W_merged.tolist(), [[2.0, 3.0]]),
    ("merged bias", b_merged.tolist(), [5.0]),
    ("identical outputs at (1,1)", (layered, merged), (10.0, 10.0)),
])

In [ ]:
import math

def relu(z):
    return max(0, z)

def tanh_act(z):
    return math.tanh(z)

def sigmoid_slope(z):
    s = 1 / (1 + math.exp(-z))
    return s * (1 - s)

run_tests([
    ("ReLU kills negatives", relu(-2.5), 0),
    ("ReLU passes positives untouched", relu(3.0), 3.0),
    ("tanh is zero-centred", tanh_act(0.0), 0.0),
    ("tanh at 1", round(tanh_act(1.0), 4), 0.7616),
    ("the slope's famous maximum", sigmoid_slope(0.0), 0.25),
    ("saturation: deaf at z=10", round(sigmoid_slope(10.0), 7), 4.54e-05),
], tol=1e-9)